In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
import datetime

# Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("ItemBased_CF_Research") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
# Load dữ liệu từ Gold
events_df = spark.table("gold_db.fact_events")

# Định nghĩa lại hàm Interaction Matrix
def get_interaction_matrix(df, ref_date):
    raw = df.groupBy("user_id", "product_id").agg(
        F.sum(F.when(F.col("event_type") == "view", 1).when(F.col("event_type") == "cart", 3).when(F.col("event_type") == "purchase", 5).otherwise(0)).alias("base_weight"),
        F.count("event_type").alias("interaction_count"),
        F.min(F.datediff(F.lit(ref_date), F.col("event_time"))).alias("days_since_last_interact")
    ).withColumn("raw_score", (F.col("base_weight") * F.log1p(F.col("interaction_count"))) / (F.lit(1) + F.lit(0.01) * F.col("days_since_last_interact")))
    
    log_df = raw.withColumn("log_score", F.log1p("raw_score"))
    stats = log_df.select(F.min("log_score").alias("min_s"), F.max("log_score").alias("max_s")).first()
    return log_df.withColumn("interaction_score", F.round(((F.col("log_score") - stats['min_s']) / (stats['max_s'] - stats['min_s'])) * 9 + 1, 4)).select("user_id", "product_id", "interaction_score")

# Chia tập dữ liệu
max_date = events_df.select(F.max("event_time")).first()[0]
cutoff = max_date - datetime.timedelta(days=30)

train_events = events_df.filter(F.col("event_time") < cutoff)
test_events = events_df.filter(F.col("event_time") >= cutoff)

train_matrix = get_interaction_matrix(train_events, cutoff).cache()
test_matrix = get_interaction_matrix(test_events, max_date).cache()


In [4]:
#  Chuẩn hóa điểm tương tác 
item_norms = train_matrix.groupBy("product_id").agg(F.sqrt(F.sum(F.pow("interaction_score", 2))).alias("norm"))

# Join dữ liệu để chuẩn bị tính toán
item_user_matrix = train_matrix.join(item_norms, "product_id")

# Self-join để tìm các cặp sản phẩm (i1, i2) được cùng 1 user tương tác
similarities = item_user_matrix.alias("i1").join(
    item_user_matrix.alias("i2"), 
    F.col("i1.user_id") == F.col("i2.user_id")
).filter(F.col("i1.product_id") < F.col("i2.product_id")) \
.groupBy(
    F.col("i1.product_id").alias("product_id_1"), 
    F.col("i2.product_id").alias("product_id_2")
).agg(
    F.sum(F.col("i1.interaction_score") * F.col("i2.interaction_score")).alias("dot_product"),
    F.max("i1.norm").alias("norm1"),
    F.max("i2.norm").alias("norm2")
).withColumn("similarity", F.col("dot_product") / (F.col("norm1") * F.col("norm2"))) \
.filter(F.col("similarity") > 0.1) \
.select("product_id_1", "product_id_2", "similarity")

# Lưu ma trận tương đồng vào Gold Layer
similarities.write.format("delta").mode("overwrite") \
    .option("path", "s3a://gold/item_similarity_matrix") \
    .saveAsTable("gold_db.item_similarity_matrix")


In [6]:
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.evaluation import RegressionEvaluator

# ĐO THỜI GIAN VÀ TÍNH ĐIỂM DỰ BÁO
start_time = time.time()

# Đọc dữ liệu từ tầng Gold 
sim_matrix = spark.table("gold_db.item_similarity_matrix")

# Sử dụng train_matrix 
user_history = train_matrix.select("user_id", "product_id", "interaction_score")

# Logic: Gợi ý sản phẩm tương đồng với lịch sử của User
# Chiều 1: User tương tác p1 -> gợi ý p2
recs_1 = user_history.join(sim_matrix, user_history.product_id == sim_matrix.product_id_1) \
    .select("user_id", F.col("product_id_2").alias("target_product"), "similarity", "interaction_score")

# Chiều 2: User tương tác p2 -> gợi ý p1
recs_2 = user_history.join(sim_matrix, user_history.product_id == sim_matrix.product_id_2) \
    .select("user_id", F.col("product_id_1").alias("target_product"), "similarity", "interaction_score")

# Tính điểm dự báo bằng Weighted Average: Sum(Sim * Score) / Sum(Sim)
recommendations = recs_1.union(recs_2) \
    .groupBy("user_id", "target_product") \
    .agg(
        (F.sum(F.col("similarity") * F.col("interaction_score")) / F.sum("similarity")).alias("prediction")
    )

training_time = time.time() - start_time

# TÍNH RMSE VÀ MAPE 
# Join với tập Test để so sánh giữa điểm thực tế (interaction_score) và điểm dự báo (prediction)
eval_df = test_matrix.select("user_id", "product_id", "interaction_score") \
    .join(recommendations, (test_matrix.user_id == recommendations.user_id) & 
          (test_matrix.product_id == recommendations.target_product))

# RMSE
rmse_evaluator = RegressionEvaluator(metricName="rmse", labelCol="interaction_score", predictionCol="prediction")
rmse_val = rmse_evaluator.evaluate(eval_df)

# MAPE
mape_val = eval_df.withColumn("abs_perc_err", 
    F.abs(F.col("interaction_score") - F.col("prediction")) / F.col("interaction_score")) \
    .select(F.avg("abs_perc_err")).first()[0] * 100

# TÍNH PRECISION, RECALL, F1 
window_spec = Window.partitionBy("user_id").orderBy(F.desc("prediction"))
top_10_recs = recommendations.withColumn("rank", F.row_number().over(window_spec)).filter(F.col("rank") <= 10)

actual_items = test_matrix.groupBy("user_id").agg(F.collect_set("product_id").alias("actual_list"))
predicted_items = top_10_recs.groupBy("user_id").agg(F.collect_set("target_product").alias("pred_list"))

comparison = actual_items.join(predicted_items, "user_id")
match_df = comparison.withColumn("hits", F.size(F.array_intersect("actual_list", "pred_list")))

precision_at_10 = match_df.select(F.avg(F.col("hits") / 10)).first()[0]
recall_at_10 = match_df.select(F.avg(F.col("hits") / F.size("actual_list"))).first()[0]
f1_val = 2 * (precision_at_10 * recall_at_10) / (precision_at_10 + recall_at_10) if (precision_at_10 + recall_at_10) > 0 else 0

# IN KẾT QUẢ TỔNG HỢP ---
print("\n" + "="*85)
print(f"{'Mô hình':<15} | {'MAPE':<10} | {'RMSE':<10} | {'Prec@10':<10} | {'Rec@10':<10} | {'F1':<10} | {'Time':<10}")
print("-" * 85)
print(f"{'Item-Based':<15} | {mape_val:>8.2f}% | {rmse_val:>10.4f} | {precision_at_10:>10.4f} | {recall_at_10:>10.4f} | {f1_val:>10.4f} | {training_time:>8.2f}s")
print("="*85 + "\n")

#  LƯU KẾT QUẢ VÀO GOLD 
# LƯU KẾT QUẢ VÀO GOLD 
top_10_recs.select(
    F.col("user_id"), 
    F.col("target_product").alias("product_id"), 
    F.col("prediction").alias("score")
) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", "s3a://gold/recommendations_itembased") \
    .saveAsTable("gold_db.recommendations_itembased")

print("SUCCESS: Đã lưu kết quả Item-based CF vào Gold Layer.")


Mô hình         | MAPE       | RMSE       | Prec@10    | Rec@10     | F1         | Time      
-------------------------------------------------------------------------------------
Item-Based      |    83.84% |     1.8817 |     0.0236 |     0.1301 |     0.0400 |     0.23s

SUCCESS: Đã lưu kết quả Item-based CF vào Gold Layer.
